# 03 -- Advanced Visualization

This notebook demonstrates advanced plotting features: side-by-side A/B comparison, handling dropped/added documents, density plots for large result sets, source provenance tracking, and the interactive Plotly backend.

In [ ]:
%matplotlib inline

import numpy as np

from rankflow import RankFlow

## A/B pipeline comparison

`RankFlow.compare()` renders two pipelines side by side so you can visually compare how documents move through different re-ranking strategies.

In [ ]:
chunk_labels = ["doc_a", "doc_b", "doc_c", "doc_d", "doc_e"]

pipeline_a = RankFlow(
    ranks=np.array([[0, 1, 2, 3, 4], [2, 0, 1, 4, 3]]),
    step_labels=["BM25", "Reranker A"],
    chunk_labels=chunk_labels,
    relevant_chunks=["doc_a", "doc_c"],
)

pipeline_b = RankFlow(
    ranks=np.array([[0, 1, 2, 3, 4], [0, 3, 1, 2, 4]]),
    step_labels=["BM25", "Reranker B"],
    chunk_labels=chunk_labels,
    relevant_chunks=["doc_a", "doc_c"],
)

RankFlow.compare(pipeline_a, pipeline_b, labels=("Reranker A", "Reranker B"))

## Handling dropped and added documents (NaN ranks)

Use `np.nan` to indicate that a document is absent at a particular step. Absent segments are drawn with a dashed line at the bottom of the plot.

In [ ]:
ranks_with_nans = np.array([
    [0,       1,       2,       np.nan],  # BM25: doc_d not retrieved
    [1,       0,       2,       3     ],  # Semantic: doc_d appears
    [np.nan,  0,       1,       2     ],  # Cross-Enc: doc_a dropped
])

rf = RankFlow(
    ranks=ranks_with_nans,
    step_labels=["BM25", "Semantic", "Cross-Encoder"],
    chunk_labels=["doc_a", "doc_b", "doc_c", "doc_d"],
    relevant_chunks=["doc_a"],
)
rf.plot()

## Density plot mode (100+ documents)

When you have many documents, individual lines become illegible. The density mode shows percentile bands (25th--75th and 10th--90th) with focus lines only for the top-K and relevant documents.

In [ ]:
rng = np.random.default_rng(42)
n_docs = 150
n_steps = 4

# Simulate ranks: each step is a permutation
ranks_large = np.column_stack([
    rng.permutation(n_docs) for _ in range(n_steps)
]).T

chunk_labels_large = [f"doc_{i:03d}" for i in range(n_docs)]
relevant = [f"doc_{i:03d}" for i in [3, 17, 42]]  # 3 known-relevant docs

rf = RankFlow(
    ranks=ranks_large,
    step_labels=["BM25", "Semantic", "Cross-Encoder", "RRF Merge"],
    chunk_labels=chunk_labels_large,
    relevant_chunks=relevant,
    density_focus_k=10,
)
rf.plot(mode="density")

## Source provenance tracking

In hybrid search, documents come from different retrieval branches (e.g., BM25 text search vs. vector search). The `source_labels` parameter lets you track and visualize the origin of each document with distinct markers and colors.

In [ ]:
ranks_hybrid = np.array([
    [0, 1, 2, 3, 4, 5],
    [2, 0, 4, 1, 3, 5],
])

rf = RankFlow(
    ranks=ranks_hybrid,
    step_labels=["Initial", "After RRF"],
    chunk_labels=["doc_a", "doc_b", "doc_c", "doc_d", "doc_e", "doc_f"],
    relevant_chunks=["doc_a", "doc_d"],
    source_labels={
        "doc_a": "text",
        "doc_b": "vector",
        "doc_c": "text",
        "doc_d": "both",
        "doc_e": "vector",
        "doc_f": "text",
    },
)
rf.plot()

## Interactive Plotly backend

Switch to the Plotly backend for interactive hover tooltips showing document name, rank, score, and delta. Requires `pip install rankflow[interactive]`.

> **Note:** If Plotly is not installed, this cell will raise an `ImportError`.

In [ ]:
rf = RankFlow(
    ranks=np.array([[0, 1, 2, 3, 4], [2, 0, 3, 1, 4], [1, 0, 2, 3, 4]]),
    step_labels=["BM25", "Semantic", "Cross-Encoder"],
    chunk_labels=["doc_a", "doc_b", "doc_c", "doc_d", "doc_e"],
    relevant_chunks=["doc_a", "doc_c"],
    scores=np.array([
        [0.95, 0.80, 0.70, 0.50, 0.30],
        [0.60, 0.92, 0.40, 0.85, 0.20],
        [0.88, 0.90, 0.75, 0.45, 0.15],
    ]),
)

try:
    rf.plot(backend="plotly")
except ImportError:
    print("Plotly not installed. Run: pip install rankflow[interactive]")

The shorthand `.iplot()` is equivalent to `.plot(backend="plotly")`.

---

**Next:** [04 -- Batch Evaluation](04_batch_evaluation.ipynb) covers evaluating retrieval quality across many queries at once.